# Script for retrieval of SMART Brick thesis data from relevant LEGO subreddits

## Importing dependencies and keywords

In [ ]:
from dependencies import *
from keywords import *
from credentials import *

In [ ]:
# creating reddit instance for calling the Reddit API through PRAW
# note that crdentials exist in credentials.py, which has not been included in the repo as it contains credentials for the author's Reddit account
reddit = praw.Reddit(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    username=USERNAME,
    password=PASSWORD,
    user_agent=USER_AGENT)

## Data retrieval

In [ ]:
# checking if csv data file already exists and loading existing post and comment IDs to avoid duplicates in .csv file
try:
    existing_df = pd.read_csv(raw_csv_filepath, encoding='utf-8-sig')
    if not existing_df.empty:
        seen_post_ids = set(existing_df[existing_df['type'] == 'post']['post_id'])
        seen_comment_ids = set(existing_df[existing_df['type'] == 'comment']['comment_id'].dropna())
        print(f"Found existing CSV with {len(existing_df)} rows at {raw_csv_filepath}")
        print(f"Existing posts: {len(seen_post_ids)}")
        print(f"Existing comments: {len(seen_comment_ids)}")
except FileNotFoundError:
    existing_df = pd.DataFrame()
    seen_post_ids = set()
    seen_comment_ids = set()
    print(f"No existing CSV found at {raw_csv_filepath} - will create new file later")

In [ ]:
# defining subreddits of interest
subreddits = {"lego", "legoleak", "legonewsandrumors", "legostarwars", "legostarwarsleaks", "legosmartbrick"}

### Scraping and processing loop

In [ ]:
results = []
keywords_found = set()
processed_posts_this_run = set()

# checking each subreddit 
for subreddit_name in subreddits:
    subreddit = reddit.subreddit(subreddit_name)

    print(f"\n{'='*60}\nSearching r/{subreddit_name}\n{'='*60}")
    # searching through all Reddit time- and sort filter endpoints
    endpoints = {}
    time_filters = ['all', 'year', 'month', 'week', 'day']
    sort_options = ['relevance', 'hot', 'top', 'new', 'comments']

    # searching for posts using each smart_brick keyword for each time- and sort filter
    for keyword in smart_brick_keywords:
        for time_filter in time_filters:
            for sort in sort_options:
                try:
                    search_results = subreddit.search(keyword, limit=None, time_filter=time_filter, sort=sort)
                    endpoints[f'search_{keyword}_{time_filter}_{sort}'] = search_results
                except Exception as e:
                    print(f"Error searching for '{keyword}' with time_filter '{time_filter}' and sort '{sort}': {e}")
                    raise

    total_posts_checked = 0
    for endpoint_name, post_list in endpoints.items():
        print(f"\nProcessing {endpoint_name}...")
        time.sleep(0.5)
        endpoint_posts = 0
        
        try:
            for attempt in range(5):
                try:
                    posts = list(post_list)
                    break
                except (prawcore.exceptions.RequestException,
                        prawcore.exceptions.ServerError,
                        prawcore.exceptions.ResponseException,
                        prawcore.exceptions.TooManyRequests) as e:
                    if attempt == 4:
                        raise
                    delay = 60 if isinstance(e, prawcore.exceptions.TooManyRequests) else 2 * (2 ** attempt)
                    print(f"{'Rate limited' if isinstance(e, prawcore.exceptions.TooManyRequests) else 'Network error'} ({e}). Retrying in {delay}s (attempt {attempt + 1}/5)...")
                    time.sleep(delay)

            for post in posts:
                # skipping posts seen within this run but not those seen in previous runs as they may have new comments
                if post.id in processed_posts_this_run:
                    continue
                processed_posts_this_run.add(post.id)
                total_posts_checked += 1
                endpoint_posts += 1

                # combine and clean post title and text
                post_text = (post.title + " " + (post.selftext or "")).lower()
                post_text_clean = re.sub(rf"[{re.escape(string.punctuation)}]", " ", post_text)

                doc = nlp(post_text_clean)
                post_lemmas = ' '.join(token.lemma_ for token in doc)

                has_smart_brick_match = any(
                    re.search(r'\b' + re.escape(lemma_kw) + r'\b', post_lemmas)
                    for lemma_kw in smart_brick_keywords)

                is_new_post = post.id not in seen_post_ids
                if has_smart_brick_match and is_new_post:
                    post_quality_matches = []
                    post_matched_sets = []
                    for set_name, keyword_set in product_quality_feature_keywords.items():
                        set_matches = []
                        for lemma_kw in keyword_set:
                            # handling currency symbols separately
                            if lemma_kw in ('$', '£', '€'):
                                if lemma_kw in post_text:
                                    set_matches.append(lemma_kw)
                            else:
                                if re.search(r'\b' + re.escape(lemma_kw) + r'\b', post_lemmas):
                                    set_matches.append(lemma_kw)
                        if set_matches:
                            post_matched_sets.append(set_name)
                            post_quality_matches.extend(set_matches)

                    keywords_found.update([kw.lower() for kw in post_quality_matches])
                    results.append({
                        'type': 'post',
                        'subreddit': subreddit_name,
                        'post_id': post.id,
                        'comment_id': None,
                        'parent_comment_id': None,
                        'author': str(post.author) if post.author else '[deleted]',
                        'created_utc': datetime.fromtimestamp(post.created_utc),
                        'post_title': post.title,
                        'text': post.title + ". " + (post.selftext or ""),
                        'lemmatized_text': post_lemmas,
                        'upvotes': post.score,
                        'is_submitter': None,
                        'matched_quality_keywords': ', '.join(post_quality_matches),
                        'matched_quality_keyword_sets': ', '.join(post_matched_sets),
                        'permalink': f"https://reddit.com{post.permalink}"})
                    seen_post_ids.add(post.id)
                    print(f"Found matching post: {post.title[:50]}...)")

                if has_smart_brick_match:
                    try:
                        for attempt in range(5):
                            try:
                                post.comments.replace_more(limit=None)
                                break
                            except (prawcore.exceptions.RequestException,
                                    prawcore.exceptions.ServerError,
                                    prawcore.exceptions.ResponseException,
                                    prawcore.exceptions.TooManyRequests) as e:
                                if attempt == 4:
                                    raise
                                delay = 60 if isinstance(e, prawcore.exceptions.TooManyRequests) else 2 * (2 ** attempt)
                                print(f"{'Rate limited' if isinstance(e, prawcore.exceptions.TooManyRequests) else 'Network error'} ({e}). Retrying in {delay}s (attempt {attempt + 1}/5)...")
                                time.sleep(delay)
                        comments = post.comments.list()
                        print(f"Found {len(comments)} comments to check")

                        comments_processed = 0
                        comments_matched = 0

                        for comment in comments:
                            if comment.id in seen_comment_ids:
                                continue

                            comments_processed += 1
                            comment_text = comment.body.lower()
                            comment_text_clean = re.sub(rf"[{re.escape(string.punctuation)}]", " ", comment_text)

                            doc = nlp(comment_text_clean)
                            comment_lemmas = ' '.join(token.lemma_ for token in doc)

                            all_quality_matches = []
                            matched_sets = []
                            for set_name, keyword_set in product_quality_feature_keywords.items():
                                set_matches = []
                                for lemma_kw in keyword_set:
                                    # handling currency symbols separately
                                    if lemma_kw in ('$', '£', '€'):
                                        if lemma_kw in comment_text:
                                            set_matches.append(lemma_kw)
                                    else:
                                        if re.search(r'\b' + re.escape(lemma_kw) + r'\b', comment_lemmas):
                                            set_matches.append(lemma_kw)
                                if set_matches:
                                    matched_sets.append(set_name)
                                    all_quality_matches.extend(set_matches)

                            if all_quality_matches:
                                keywords_found.update([kw.lower() for kw in all_quality_matches])
                                comments_matched += 1

                                parent_id = comment.parent_id
                                parent_comment_id = parent_id[3:] if parent_id.startswith('t1_') else None

                                results.append({
                                    'type': 'comment',
                                    'subreddit': subreddit_name,
                                    'post_id': post.id,
                                    'comment_id': comment.id,
                                    'parent_comment_id': parent_comment_id,
                                    'author': str(comment.author) if comment.author else '[deleted]',
                                    'created_utc': datetime.fromtimestamp(comment.created_utc),
                                    'post_title': post.title,
                                    'text': comment.body,
                                    'lemmatized_text': comment_lemmas,
                                    'upvotes': comment.score,
                                    'is_submitter': comment.is_submitter,
                                    'matched_quality_keywords': ', '.join(all_quality_matches),
                                    'matched_quality_keyword_sets': ', '.join(matched_sets),
                                    'permalink': f"https://reddit.com{comment.permalink}"})
                                seen_comment_ids.add(comment.id)
                                print(f"Comment match #{comments_matched} by {comment.author} (quality: {len(all_quality_matches)})")
                        print(f"Processed {comments_processed} comments, found {comments_matched} matches")
                    except Exception as e:
                        print(f"Error processing comments: {e}")
                        raise
        except Exception as e:
            print(f"Error processing {endpoint_name}: {e}")
            raise

        print(f"Checked {endpoint_posts} posts from {endpoint_name}")

    print(f"Total unique posts checked in r/{subreddit_name}: {total_posts_checked}")
    print(f"Total results so far: {len(results)}")

### Saving retrieved data

In [ ]:
print(f"Data retrieval done, creating df based on results...")
new_df = pd.DataFrame(results)
print(f"{len(new_df)} new results found")

In [ ]:
# normalizing text columns for .csv/.tsv human readability
def normalize_text(s):
    if isinstance(s, str):
        s = re.sub(r'[\r\n\t\v\f\xa0]+', ' ', s)
        s = re.sub(r' {2,}', ' ', s)
        s = s.replace('"', "'")
        return s.strip()
    return s

text_columns = ['post_title', 'text', "lemmatized_text"]
for col in text_columns:
    if col in new_df.columns:
        new_df[col] = new_df[col].apply(normalize_text)

In [ ]:
# saving to .csv and .tsv 
if not new_df.empty:
    # .csv
    new_df.to_csv(
        raw_csv_filepath,
        mode='a' if os.path.exists(raw_csv_filepath) else 'w',
        header=not os.path.exists(raw_csv_filepath),
        index=False,
        encoding='utf-8-sig',
        quoting=csv.QUOTE_ALL,
        escapechar='\\',
        lineterminator='\n')

    # .tsv
    new_df.to_csv(
        raw_tsv_filepath,
        mode='a' if os.path.exists(raw_tsv_filepath) else 'w',
        header=not os.path.exists(raw_tsv_filepath),
        index=False,
        sep='\t',
        encoding='utf-8-sig',
        quoting=csv.QUOTE_ALL,
        escapechar='\\',
        lineterminator='\n')

    print(f"Saved {len(new_df)} new entries to {raw_csv_filepath} and {raw_tsv_filepath}")
    
else:
    print("No new entries found in this run")